# TRACED — Usage Examples

**TRACED** (Threshold-based Recursive Adaptive Clustering with Ellipsoidal Decision) เป็น online classifier แบบ single-pass ที่เรียนรู้ข้อมูลทีละ sample โดยไม่เก็บ training data ไว้

แต่ละ class จะถูกแทนด้วยกลุ่มของ **hyperellipsoid neurons** ที่ขยายและรวมตัวกันแบบ incremental

### Hyperparameters หลัก

| Parameter | Default | ความหมาย |
|-----------|---------|-----------|
| `method` | `'overlap-outside'` | เลือก region ที่จะ resolve: `'overlap'`, `'outside'`, หรือ `'overlap-outside'` |
| `reduce_dims` | `0` | จำนวน PCA dimension ที่ตัดออกตอน overlap resolution (0 = ปิด) |
| `distance_metric` | `'boundary'` | วิธีวัดระยะ: `'boundary'` หรือ `'center'` |
| `N0` | `3` | จำนวน sample ขั้นต่ำที่ neuron ต้องมีก่อนจะโหวตใน predict |
| `delta` | `2` | ตัวคูณ distance threshold ตอนสร้าง neuron ใหม่ |
| `r` | `√(2π)` | scale factor ของ ellipsoid |
| `alpha` | `0` | smoothing ของ displacement (0 = ไม่จำ) |
| `beta` | `0` | smoothing ของ expansion (0 = ไม่จำ) |
| `width_parameter` | `1` | blend ระหว่าง PCA-width และ shift-width |
| `norm` | `2` | Minkowski norm (2 = Euclidean) |
| `threshold` | `15` | angle threshold สำหรับจับคู่ PCA axes (องศา) |

In [1]:
import numpy as np
from sklearn.datasets import load_iris, load_digits, make_classification
from sklearn.metrics import accuracy_score
from sklearn.base import clone

from spdal import TRACED

## 1. Basic Batch Usage

เรียนรู้ข้อมูลทั้งหมดในรอบเดียวด้วย `partial_fit` แล้ว `predict`

In [2]:
X, y = load_iris(return_X_y=True)
X_train, X_test = X[:120], X[120:]
y_train, y_test = y[:120], y[120:]

clf = TRACED()
clf.partial_fit(X_train, y_train, classes=np.unique(y))
y_pred = clf.predict(X_test)

print(f"Classes learned : {clf.classes_}")
print(f"Neurons created : {len(clf.neuron_list)}")
print(f"Accuracy        : {accuracy_score(y_test, y_pred):.2%}")
print(f"Predictions     : {y_pred.tolist()}")

Classes learned : [0 1 2]
Neurons created : 7
Accuracy        : 96.67%
Predictions     : [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]


## 2. Neuron Inspection

`clf.neuron_list` คือ `list[dict]` — แต่ละ neuron เก็บ `center`, `width`, `cov`, `eig_component`, `n`, `displacement`, `expansion`

In [3]:
print(f"{'#':<4} {'class':<7} {'n':>5}   {'center':^32}   {'width':^32}")
print("-" * 85)
for i, n in enumerate(clf.neuron_list):
    center_str = str(np.round(n['center'], 2))
    width_str  = str(np.round(n['width'],  3))
    print(f"{i:<4} {str(n['y']):<7} {n['n']:>5}   {center_str:<32}   {width_str:<32}")

#    class       n                center                             width              
-------------------------------------------------------------------------------------
0    0          46   [5.   3.43 1.48 0.25]              [0.995 0.459 0.381 0.237]       
1    0           2   [4.4  2.65 1.2  0.2 ]              [0.979 0.    0.    0.   ]       
2    0           2   [5.75 4.2  1.35 0.3 ]              [0.686 0.    0.    0.   ]       
3    1          44   [6.04 2.82 4.38 1.37]              [1.359 0.681 0.57  0.25 ]       
4    1           6   [5.2  2.37 3.38 1.02]              [0.848 0.568 0.184 0.044]       
5    2          19   [6.65 2.94 5.72 2.06]              [2.14  0.911 0.543 0.341]       
6    2           1   [4.9 2.5 4.5 1.7]                  [0. 0. 0. 0.]                   


## 3. Incremental Learning — 10 Chunks (Phishing dataset)

ใช้ `partial_fit` ซ้ำ ๆ กับข้อมูล stream ทีละ chunk  
Phishing dataset จาก river: 1,250 samples, 9 features, 2 classes (phishing vs legit)

> ติดตั้ง river: `pip install -e ".[experiments]"`

In [4]:
from river.datasets import Phishing

data = Phishing()
X_ph = np.array([np.array(list(s[0].values())) for s in data])
y_ph = np.array([int(s[1]) for s in data])
classes_ph = np.unique(y_ph)

X_train_ph, X_test_ph = X_ph[:1000], X_ph[1000:]
y_train_ph, y_test_ph = y_ph[:1000], y_ph[1000:]
chunks = [(X_train_ph[i:i+100], y_train_ph[i:i+100]) for i in range(0, 1000, 100)]

print(f"Dataset: {X_ph.shape[0]} samples, {X_ph.shape[1]} features, classes={classes_ph}")
print(f"Train: {len(X_train_ph)}  |  Test: {len(X_test_ph)}  |  Chunks: {len(chunks)} × 100\n")

clf_ph = TRACED(method="overlap-outside", reduce_dims=1)

print(f"{'Chunk':>7}  {'Neurons':>7}  {'Accuracy':>9}  {'Overlap':>8}  {'Outside':>8}")
print("-" * 50)
for i, (Xc, yc) in enumerate(chunks):
    clf_ph.partial_fit(Xc, yc, classes=classes_ph)
    acc = accuracy_score(y_test_ph, clf_ph.predict(X_test_ph))
    print(f"{i+1:>5}/10  {len(clf_ph.neuron_list):>7}  {acc:>9.2%}"
          f"  {clf_ph.count_overlap:>8}  {clf_ph.count_outside:>8}")

Dataset: 1250 samples, 9 features, classes=[0 1]
Train: 1000  |  Test: 250  |  Chunks: 10 × 100

  Chunk  Neurons   Accuracy   Overlap   Outside
--------------------------------------------------
    1/10        2     90.00%         0       192
    2/10        2     92.00%         0       370
    3/10        2     92.00%         0       548
    4/10        2     90.80%         0       714
    5/10        2     91.60%         0       881
    6/10        2     90.40%         0      1040
    7/10        2     90.40%         0      1199
    8/10        2     90.40%         0      1362
    9/10        2     90.40%         0      1526
   10/10        2     90.80%         0      1690


## 4. Resolution Methods: `overlap`, `outside`, `overlap-outside`

TRACED แบ่ง test point ออกเป็น 3 zone:

```
              boundary
                 │
   outside  [===│=== neuron ===│===]  outside
                 │
```

- **overlap** — point อยู่ภายใน ellipsoid ของสอง class พร้อมกัน (distance ≤ 0 ทั้งคู่) → ใช้ PCA projection ช่วย
- **outside** — point อยู่นอก ellipsoid ของทุก class (distance > 0 ทั้งคู่) → ใช้ displacement + expansion ช่วย
- **overlap-outside** — ทั้งสอง (default)

In [5]:
X_bin, y_bin = make_classification(
    n_samples=500, n_features=10, n_informative=5, random_state=42
)
X_tr, X_te = X_bin[:400], X_bin[400:]
y_tr, y_te = y_bin[:400], y_bin[400:]

print(f"{'method':<22}  {'neurons':>7}  {'accuracy':>9}  {'overlap':>8}  {'outside':>8}")
print("-" * 60)
for method in ["overlap-outside", "overlap", "outside"]:
    clf_m = TRACED(method=method, reduce_dims=2)
    clf_m.partial_fit(X_tr, y_tr, classes=np.unique(y_bin))
    preds = clf_m.predict(X_te)
    print(f"{method:<22}  {len(clf_m.neuron_list):>7}  {accuracy_score(y_te, preds):>9.2%}"
          f"  {clf_m.count_overlap:>8}  {clf_m.count_outside:>8}")

method                  neurons   accuracy   overlap   outside
------------------------------------------------------------
overlap-outside               2     89.00%         4        59
overlap                       2     89.00%         4         0
outside                       2     91.00%         0        59


## 5. Distance Metric: `boundary` vs `center`

- **`boundary`** (default) — วัดระยะห่างจาก boundary ของ ellipsoid (ค่า < 0 = อยู่ใน, > 0 = อยู่นอก)
- **`center`** — วัดระยะเทียบกับ center แบบ normalized (ค่า < 0 = อยู่ใน)

In [6]:
X, y = load_iris(return_X_y=True)
X_tr, X_te, y_tr = X[:120], X[120:], y[:120]

print(f"{'distance_metric':<14}  {'neurons':>7}  {'accuracy':>9}")
print("-" * 35)
for metric in ["boundary", "center"]:
    clf_d = TRACED(distance_metric=metric)
    clf_d.partial_fit(X_tr, y_tr, classes=np.unique(y))
    acc = accuracy_score(y[120:], clf_d.predict(X_te))
    print(f"{metric:<14}  {len(clf_d.neuron_list):>7}  {acc:>9.2%}")

distance_metric  neurons   accuracy
-----------------------------------
boundary              7     96.67%
center                7    100.00%


## 6. Effect of `N0` — Neuron Eligibility Threshold

`N0` กำหนดว่า neuron ต้องสะสม sample อย่างน้อยกี่ตัวถึงจะมีสิทธิ์โหวตใน `predict`  
ช่วยกรอง neuron ที่เพิ่งสร้างและยังไม่ mature ออกไปก่อน

In [7]:
print(f"{'N0':>4}  {'total neurons':>14}  {'eligible':>9}  {'accuracy':>9}")
print("-" * 42)
for n0 in [1, 3, 5, 10]:
    clf_n = TRACED(N0=n0)
    clf_n.partial_fit(X_tr, y_tr, classes=np.unique(y))
    eligible = sum(1 for n in clf_n.neuron_list if n['n'] >= n0)
    acc = accuracy_score(y[120:], clf_n.predict(X_te))
    print(f"{n0:>4}  {len(clf_n.neuron_list):>14}  {eligible:>9}  {acc:>9.2%}")

  N0   total neurons   eligible   accuracy
------------------------------------------


   1               8          8    100.00%
   3               7          4     96.67%
   5               7          4     96.67%
  10               7          3     96.67%


## 7. Adaptive Displacement & Expansion: `alpha`, `beta`

- **`alpha`** — smoothing ของ `displacement` (การ track ทิศทางที่ neuron เคลื่อนที่)  
  - `0` = ไม่จำทิศทางเก่าเลย (default)  
  - `> 0` = ผสม momentum จาก step ก่อนหน้า ช่วยรองรับ outside points ได้ดีขึ้น

- **`beta`** — smoothing ของ `expansion` (การขยาย width ตามทิศทางที่ neuron โต)  
  - `0` = ไม่สะสม expansion (default)  
  - `> 0` = ขยาย boundary ในทิศทางที่ neuron เคยโตมา

In [8]:
configs = [
    {"alpha": 0.0, "beta": 0.0},
    {"alpha": 0.3, "beta": 0.0},
    {"alpha": 0.0, "beta": 0.3},
    {"alpha": 0.3, "beta": 0.3},
]

print(f"{'alpha':>6}  {'beta':>6}  {'neurons':>7}  {'accuracy':>9}")
print("-" * 38)
for cfg in configs:
    clf_ab = TRACED(**cfg, method="overlap-outside", reduce_dims=1)
    clf_ab.partial_fit(X_tr, y_tr, classes=np.unique(y))
    acc = accuracy_score(y[120:], clf_ab.predict(X_te))
    print(f"{cfg['alpha']:>6.1f}  {cfg['beta']:>6.1f}  {len(clf_ab.neuron_list):>7}  {acc:>9.2%}")

 alpha    beta  neurons   accuracy
--------------------------------------
   0.0     0.0        7     96.67%
   0.3     0.0        7    100.00%
   0.0     0.3        7    100.00%
   0.3     0.3        7    100.00%


## 8. High-dimensional Data — Digits Dataset

Digits: 1,797 samples, **64 features**, 10 classes  
ใช้ `reduce_dims` เพื่อตัด PCA dimension ที่ overlap กันออกตอน resolution

In [9]:
X_dg, y_dg = load_digits(return_X_y=True)
X_tr_dg, X_te_dg = X_dg[:1200], X_dg[1200:]
y_tr_dg, y_te_dg = y_dg[:1200], y_dg[1200:]

print(f"Dataset: {X_dg.shape[0]} samples, {X_dg.shape[1]} features, {len(np.unique(y_dg))} classes\n")
print(f"{'reduce_dims':>12}  {'neurons':>7}  {'accuracy':>9}  {'overlap':>8}  {'outside':>8}")
print("-" * 55)
for rd in [0, 2, 4, 8]:
    clf_dg = TRACED(method="overlap-outside", reduce_dims=rd, N0=5)
    clf_dg.partial_fit(X_tr_dg, y_tr_dg, classes=np.unique(y_dg))
    acc = accuracy_score(y_te_dg, clf_dg.predict(X_te_dg))
    print(f"{rd:>12}  {len(clf_dg.neuron_list):>7}  {acc:>9.2%}"
          f"  {clf_dg.count_overlap:>8}  {clf_dg.count_outside:>8}")

Dataset: 1797 samples, 64 features, 10 classes

 reduce_dims  neurons   accuracy   overlap   outside
-------------------------------------------------------
           0       10     89.95%         0       597


           2       10     89.95%         0       597


           4       10     89.95%         0       597


           8       10     89.95%         0       597


## 9. sklearn Compatibility

TRACED รับ `BaseEstimator` จาก sklearn ทำให้ใช้ได้กับ:
- `get_params()` / `set_params()` — ดู/เซต hyperparameters
- `clone()` — สร้าง instance ใหม่ที่ยัง unfitted แต่มี params เหมือนกัน

In [10]:
original = TRACED(method="overlap", reduce_dims=2, N0=5, alpha=0.2)
original.partial_fit(X_tr, y_tr, classes=np.unique(y))

print("get_params():")
for k, v in original.get_params().items():
    print(f"  {k:<20} = {v}")

print()
cloned = clone(original)
print(f"clone() — unfitted copy:")
print(f"  neuron_list empty : {len(cloned.neuron_list) == 0}")
print(f"  has classes_      : {hasattr(cloned, 'classes_')}")
print(f"  params preserved  : {cloned.get_params() == original.get_params()}")

cloned.partial_fit(X_tr, y_tr, classes=np.unique(y))
acc = accuracy_score(y[120:], cloned.predict(X_te))
print(f"\nclone() after fit:")
print(f"  neurons   : {len(cloned.neuron_list)}")
print(f"  accuracy  : {acc:.2%}")

get_params():
  N0                   = 5
  alpha                = 0.2
  beta                 = 0
  delta                = 2
  distance_metric      = boundary
  epsilon              = 1e-10
  method               = overlap
  norm                 = 2
  r                    = 2.5066282746310002
  reduce_dims          = 2
  threshold            = 15
  width_parameter      = 1

clone() — unfitted copy:
  neuron_list empty : True
  has classes_      : False
  params preserved  : True

clone() after fit:
  neurons   : 7
  accuracy  : 96.67%


## 10. Continuous Streaming — ติดตาม Accuracy ทุก Chunk

จำลอง data stream จริง: ข้อมูลมาทีละ chunk ต่อเนื่อง  
`count_overlap` และ `count_outside` สะสมตลอด session ไม่ reset

In [11]:
X_s, y_s = make_classification(n_samples=1000, n_features=8, random_state=0)
X_te_s, y_te_s = X_s[800:], y_s[800:]
chunks_s = [(X_s[i:i+100], y_s[i:i+100]) for i in range(0, 800, 100)]

clf_s = TRACED(method="overlap-outside", reduce_dims=1)

print(f"{'Chunk':>7}  {'Neurons':>7}  {'Accuracy':>9}  {'Overlap':>8}  {'Outside':>8}")
print("-" * 50)
for i, (Xc, yc) in enumerate(chunks_s):
    clf_s.partial_fit(Xc, yc, classes=np.array([0, 1]))
    acc = accuracy_score(y_te_s, clf_s.predict(X_te_s))
    print(f"{i+1:>5}/8   {len(clf_s.neuron_list):>7}  {acc:>9.2%}"
          f"  {clf_s.count_overlap:>8}  {clf_s.count_outside:>8}")

  Chunk  Neurons   Accuracy   Overlap   Outside
--------------------------------------------------
    1/8         2     90.50%         0        83
    2/8         2     92.50%        14       156
    3/8         2     93.00%        23       225
    4/8         2     93.00%        34       291
    5/8         2     93.00%        46       357
    6/8         2     93.50%        58       419
    7/8         2     93.50%        69       485
    8/8         2     92.50%        80       551
